# Identificacion FOPDT con Ziegler-Nichols y Smith

Esta notebook organiza y explica el proceso de identificacion de un modelo FOPDT usando:
- Regresion polinomica para ubicar el punto de inflexion
- Metodo de Ziegler-Nichols (curva de reaccion)
- Metodo de Smith (28.3% y 63.2%)

Tambien se generan tres graficas: ajuste por tangente, grafica de Smith y comparacion final de modelos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1) Cargar y preparar datos
Se leen los datos del archivo, se limpian nombres de columnas y se extraen las variables de tiempo, entrada y salida.

In [ ]:
file_path = 'data_Q2.txt'
# Usamos latin1 porque el archivo viene de Windows/ANSI con simbolos de grado
data = pd.read_csv(file_path, skipinitialspace=True, encoding='latin1')

# Limpiar nombres de columnas
data.columns = [c.strip() for c in data.columns]

t = data['Tiempo (s)'].values
u = data['Cal2 (%)'].values
y = data['Temp2 (°C)'].values

data.head()

## 2) Detectar el escalon de entrada
Se identifica el primer incremento positivo en la senal de entrada y se calculan valores base.

In [ ]:
idx_step = np.where(np.diff(u) > 0)[0][0] + 1
t_step = t[idx_step]
u_initial = u[0]
u_final = u[-1]
delta_u = u_final - u_initial

print(f'Indice del escalon: {idx_step}')
print(f'Tiempo del escalon t0: {t_step:.4f} s')
print(f'Delta_u: {delta_u:.4f} %')

## 3) Regresion polinomica y punto de inflexion
Se ajusta un polinomio grado 6 para suavizar la respuesta y se usa su derivada para encontrar la pendiente maxima.

In [ ]:
coefs = np.polyfit(t, y, 6)
poly_func = np.poly1d(coefs)
y_fit = poly_func(t)

poly_der = np.polyder(poly_func)
pendientes = poly_der(t)

idx_max_pendiente = idx_step + np.argmax(pendientes[idx_step:])
t_inflexion = t[idx_max_pendiente]
y_inflexion = y_fit[idx_max_pendiente]
m_max = pendientes[idx_max_pendiente]

print(f'Pendiente maxima: {m_max:.6f}')
print(f'Punto de inflexion: t={t_inflexion:.4f} s, y={y_inflexion:.4f}')

## 4) Parametros por Ziegler-Nichols
Con la tangente en el punto de inflexion se obtienen $K$, $\theta$ y $\tau$.

In [ ]:
y_initial = np.mean(y[:idx_step])
y_final = np.mean(y[-20:])
K = (y_final - y_initial) / delta_u

t_base = ((y_initial - y_inflexion) / m_max) + t_inflexion
theta = t_base - t_step

t_final_interseccion = ((y_final - y_inflexion) / m_max) + t_inflexion
tau = t_final_interseccion - t_base

print('--- Parametros por Regresion Polinomica ---')
print(f'Ganancia (K): {K:.4f}')
print(f'Tiempo Muerto (theta): {theta:.4f} s')
print(f'Constante de Tiempo (tau): {tau:.4f} s')

## 5) Parametros por Smith (28.3% y 63.2%)
Se calcula el tiempo de cruce en los niveles 28.3% y 63.2% para estimar $\tau$ y $\theta$.

In [ ]:
def first_crossing_time(t_arr, y_arr, y_level, start_idx, direction=1):
    for i in range(start_idx + 1, len(t_arr)):
        y_prev = y_arr[i - 1]
        y_curr = y_arr[i]

        if direction >= 0:
            crossed = (y_prev <= y_level <= y_curr)
        else:
            crossed = (y_prev >= y_level >= y_curr)

        if crossed:
            if y_curr == y_prev:
                return t_arr[i]
            frac = (y_level - y_prev) / (y_curr - y_prev)
            return t_arr[i - 1] + frac * (t_arr[i] - t_arr[i - 1])

    return np.nan

direction = 1 if (y_final - y_initial) >= 0 else -1
y_28 = y_initial + 0.283 * (y_final - y_initial)
y_63 = y_initial + 0.632 * (y_final - y_initial)

t1 = first_crossing_time(t, y_fit, y_28, idx_step, direction)
t2 = first_crossing_time(t, y_fit, y_63, idx_step, direction)

if np.isnan(t1) or np.isnan(t2) or t2 <= t1:
    theta_smith = np.nan
    tau_smith = np.nan
else:
    tau_smith = max(1.5 * (t2 - t1), 1e-6)
    theta_smith = max((1.5 * t1 - 0.5 * t2) - t_step, 0.0)

print('--- Parametros por Metodo de Smith ---')
print(f't1 (28.3%): {t1:.4f} s')
print(f't2 (63.2%): {t2:.4f} s')
print(f'Tiempo Muerto Smith (theta): {theta_smith:.4f} s')
print(f'Constante de Tiempo Smith (tau): {tau_smith:.4f} s')

if not np.isnan(theta_smith) and not np.isnan(tau_smith):
    tt_smith = t - (t_step + theta_smith)
    y_smith = y_initial + K * delta_u * np.where(tt_smith > 0, 1.0 - np.exp(-tt_smith / tau_smith), 0.0)
else:
    y_smith = np.full_like(t, np.nan, dtype=float)

## 6) Grafica de Ziegler-Nichols
Se muestra la regresion y la tangente usada para estimar los parametros por curva de reaccion.

In [ ]:
plt.figure(figsize=(12, 7))
plt.plot(t, y, 'k.', alpha=0.2, label='Datos Originales')
plt.plot(t, y_fit, 'b-', label='Regresion Polinomica (Grado 6)')

t_tangente = np.linspace(t_base, t_final_interseccion, 100)
y_tangente = m_max * (t_tangente - t_inflexion) + y_inflexion
plt.plot(t_tangente, y_tangente, 'r--', lw=2, label='Tangente en Inflexion')

plt.axhline(y_initial, color='green', linestyle=':', label='Valor Inicial')
plt.axhline(y_final, color='red', linestyle=':', label='Valor Final')
plt.axvline(t_step, color='black', alpha=0.5, label='Inicio Escalon')

plt.title('Ziegler-Nichols: Analisis por Regresion')
plt.xlabel('Tiempo (s)')
plt.ylabel('Temperatura (°C)')
plt.legend()
plt.grid(True)
plt.show()

## 7) Grafica de Smith
Visualizacion del modelo obtenido con los niveles caracteristicos del metodo de Smith.

In [ ]:
plt.figure(figsize=(12, 7))
plt.plot(t, y, 'k.', alpha=0.2, label='Datos Originales')
plt.plot(t, y_fit, 'b-', lw=2, label='Regresion Polinomica (Grado 6)')

if not np.all(np.isnan(y_smith)):
    plt.plot(t, y_smith, 'm-', lw=2.5, label='Modelo FOPDT por Smith')

if not np.isnan(t1):
    plt.axvline(t1, color='orange', linestyle='--', alpha=0.7, label='t1 (28.3%)')
if not np.isnan(t2):
    plt.axvline(t2, color='red', linestyle='--', alpha=0.7, label='t2 (63.2%)')
if not np.isnan(theta_smith):
    plt.axvline(t_step + theta_smith, color='purple', linestyle=':', alpha=0.9, label='t0 + theta Smith')

plt.axhline(y_28, color='orange', linestyle=':', alpha=0.8, label='Nivel 28.3%')
plt.axhline(y_63, color='red', linestyle=':', alpha=0.8, label='Nivel 63.2%')
plt.axvline(t_step, color='black', alpha=0.5, label='Inicio Escalon')

plt.title('Metodo de Smith: Identificacion FOPDT')
plt.xlabel('Tiempo (s)')
plt.ylabel('Temperatura (°C)')
plt.legend()
plt.grid(True)
plt.show()

## 8) Comparacion final de modelos
Comparacion directa entre FOPDT estimado por Ziegler-Nichols y por Smith frente a los datos medidos.

In [ ]:
if not np.isnan(theta) and not np.isnan(tau):
    tt_zn = t - (t_step + theta)
    y_zn = y_initial + K * delta_u * np.where(tt_zn > 0, 1.0 - np.exp(-tt_zn / tau), 0.0)
else:
    y_zn = np.full_like(t, np.nan, dtype=float)

plt.figure(figsize=(12, 7))
plt.plot(t, y, 'k.', alpha=0.25, label='Datos Originales')

if not np.all(np.isnan(y_zn)):
    plt.plot(t, y_zn, 'c--', lw=2.5,
             label=f'FOPDT Ziegler-Nichols (tau={tau:.2f}, theta={theta:.2f})')

if not np.all(np.isnan(y_smith)):
    plt.plot(t, y_smith, 'm-', lw=2.5,
             label=f'FOPDT Smith (tau={tau_smith:.2f}, theta={theta_smith:.2f})')

plt.axvline(t_step, color='black', alpha=0.5, linestyle=':', label='Inicio Escalon')
plt.title('Comparacion de Modelos FOPDT: Ziegler-Nichols vs Smith')
plt.xlabel('Tiempo (s)')
plt.ylabel('Temperatura (°C)')
plt.legend()
plt.grid(True)
plt.show()

## 9) Sintonia PID (Kp, Ti, Td) y optimizacion
A partir del modelo FOPDT identificado, se calcula una sintonia inicial PID con Ziegler-Nichols y luego se optimiza minimizando una funcion de costo que pondera:

- Tiempo de asentamiento (`Ts`)
- Sobreimpulso (`Mp`)
- Error absoluto integrado (`IAE`)
- Error en regimen permanente (`ess`)

> Nota: si `scipy` no esta disponible, se usa una busqueda aleatoria acotada para no detener el flujo del notebook.

In [ ]:
# Base de modelo para sintonia: prioriza Smith si existe, si no usa ZN
if not np.isnan(tau_smith) and not np.isnan(theta_smith):
    tau_base = float(tau_smith)
    theta_base = float(theta_smith)
    metodo_base = 'Smith'
else:
    tau_base = float(tau)
    theta_base = float(theta)
    metodo_base = 'Ziegler-Nichols'

# Parametros iniciales por Ziegler-Nichols (curva de reaccion)
# C(s) = Kp * (1 + 1/(Ti s) + Td s)
eps = 1e-9
if abs(K * theta_base) < eps:
    raise ValueError('No es posible calcular Kp inicial: K*theta ~ 0.')

Kp_ini = 1.2 * tau_base / (K * max(theta_base, eps))
Ti_ini = max(2.0 * theta_base, 1e-6)
Td_ini = max(0.5 * theta_base, 0.0)

# Simulacion discreta de lazo cerrado para proceso FOPDT con retardo
# tau * dy/dt = -y + K * u_delayed
# e = r - y; u = Kp * (e + int(e)/Ti + Td * de/dt)
def simulate_closed_loop_fopdt_pid(Kp, Ti, Td, K_proc, tau_proc, theta_proc,
                                   t_end=400.0, dt=1.0, r=1.0):
    n = int(t_end / dt) + 1
    t_sim = np.linspace(0.0, t_end, n)

    y_dev = np.zeros(n)
    u_dev = np.zeros(n)
    e_prev = 0.0
    e_int = 0.0

    n_delay = max(int(round(theta_proc / dt)), 0)
    delay_buffer = [0.0 for _ in range(n_delay + 1)]

    for k in range(1, n):
        e = r - y_dev[k - 1]
        e_int += e * dt
        e_der = (e - e_prev) / dt

        u_now = Kp * (e + (e_int / max(Ti, 1e-9)) + Td * e_der)
        u_dev[k] = u_now

        delay_buffer.append(u_now)
        u_del = delay_buffer.pop(0)

        dy = (-y_dev[k - 1] + K_proc * u_del) / max(tau_proc, 1e-9)
        y_dev[k] = y_dev[k - 1] + dt * dy

        e_prev = e

    return t_sim, y_dev, u_dev


def performance_metrics(t_sim, y_dev, r):
    e = r - y_dev

    if abs(r) < 1e-9:
        mp = 0.0
        band = 0.02
    else:
        mp = max(0.0, (np.max(y_dev) - r) / abs(r) * 100.0)
        band = 0.02 * abs(r)

    outside = np.where(np.abs(y_dev - r) > band)[0]
    Ts = float(t_sim[outside[-1]]) if len(outside) > 0 else 0.0
    IAE = float(np.trapz(np.abs(e), t_sim))
    ess = float(abs(e[-1]))

    return {'Ts': Ts, 'Mp': mp, 'IAE': IAE, 'ess': ess}


def cost_pid(x, K_proc, tau_proc, theta_proc, r=1.0):
    Kp_, Ti_, Td_ = x
    if (Kp_ <= 0) or (Ti_ <= 0) or (Td_ < 0):
        return np.inf

    t_sim, y_dev, _ = simulate_closed_loop_fopdt_pid(
        Kp_, Ti_, Td_, K_proc, tau_proc, theta_proc, t_end=400.0, dt=1.0, r=r
    )

    if np.any(np.isnan(y_dev)) or np.max(np.abs(y_dev)) > 5 * abs(r) + 10:
        return np.inf

    m = performance_metrics(t_sim, y_dev, r)

    # Funcion de costo ponderada
    J = 1.0 * m['Ts'] + 0.8 * m['Mp'] + 0.2 * m['IAE'] + 30.0 * m['ess']
    return float(J)


# Referencia en variables de desviacion: 1.0
r_step = 1.0
x0 = np.array([max(Kp_ini, 1e-6), max(Ti_ini, 1e-6), max(Td_ini, 0.0)], dtype=float)

# Limites razonables alrededor de la sintonia inicial
bnds = [
    (0.2 * x0[0], 5.0 * x0[0]),
    (0.2 * x0[1], 5.0 * x0[1]),
    (0.0, 5.0 * max(x0[2], 1e-3)),
]

used_scipy = False
try:
    from scipy.optimize import minimize

    sol = minimize(
        lambda z: cost_pid(z, K, tau_base, theta_base, r=r_step),
        x0=x0,
        method='L-BFGS-B',
        bounds=bnds,
    )

    if sol.success and np.isfinite(sol.fun):
        x_opt = sol.x
        used_scipy = True
    else:
        x_opt = x0.copy()
except Exception:
    # Fallback: busqueda aleatoria acotada (sin scipy)
    rng = np.random.default_rng(42)
    x_opt = x0.copy()
    best_cost = cost_pid(x_opt, K, tau_base, theta_base, r=r_step)

    for _ in range(1200):
        cand = np.array([
            rng.uniform(bnds[0][0], bnds[0][1]),
            rng.uniform(bnds[1][0], bnds[1][1]),
            rng.uniform(bnds[2][0], bnds[2][1]),
        ], dtype=float)
        c = cost_pid(cand, K, tau_base, theta_base, r=r_step)
        if c < best_cost:
            best_cost = c
            x_opt = cand

Kp_opt, Ti_opt, Td_opt = x_opt

# Evaluacion inicial vs optimizado
t_i, y_i, u_i = simulate_closed_loop_fopdt_pid(Kp_ini, Ti_ini, Td_ini, K, tau_base, theta_base, t_end=400.0, dt=1.0, r=r_step)
t_o, y_o, u_o = simulate_closed_loop_fopdt_pid(Kp_opt, Ti_opt, Td_opt, K, tau_base, theta_base, t_end=400.0, dt=1.0, r=r_step)

met_i = performance_metrics(t_i, y_i, r_step)
met_o = performance_metrics(t_o, y_o, r_step)

df_pid = pd.DataFrame([
    {
        'Caso': 'Inicial',
        'Metodo base modelo': metodo_base,
        'Kp': Kp_ini,
        'Ti': Ti_ini,
        'Td': Td_ini,
        'Ts (s)': met_i['Ts'],
        'Mp (%)': met_i['Mp'],
        'IAE': met_i['IAE'],
        'ess': met_i['ess'],
    },
    {
        'Caso': 'Optimizado',
        'Metodo base modelo': metodo_base,
        'Kp': Kp_opt,
        'Ti': Ti_opt,
        'Td': Td_opt,
        'Ts (s)': met_o['Ts'],
        'Mp (%)': met_o['Mp'],
        'IAE': met_o['IAE'],
        'ess': met_o['ess'],
    },
])

print(f'Metodo base para sintonia: {metodo_base}')
print('Optimizacion con scipy:' + (' si' if used_scipy else ' no (fallback aleatorio)'))
display(df_pid.round(4))

In [ ]:
# Visualizacion de respuesta en lazo cerrado: inicial vs optimizado
plt.figure(figsize=(12, 6))
plt.plot(t_i, np.ones_like(t_i) * r_step, 'k:', lw=2, label='Referencia')
plt.plot(t_i, y_i, 'tab:blue', lw=2, label='PID inicial')
plt.plot(t_o, y_o, 'tab:green', lw=2.5, label='PID optimizado')

plt.title('Respuesta en Lazo Cerrado (modelo FOPDT)')
plt.xlabel('Tiempo (s)')
plt.ylabel('Salida normalizada')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(12, 4))
plt.plot(t_i, u_i, 'tab:blue', alpha=0.8, label='Accion de control inicial')
plt.plot(t_o, u_o, 'tab:green', alpha=0.8, label='Accion de control optimizada')
plt.title('Accion de Control PID')
plt.xlabel('Tiempo (s)')
plt.ylabel('u (desviacion)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 10) Error de ajuste: datos vs Ziegler-Nichols y datos vs Smith
Se calculan metricas de error para comparar que modelo FOPDT representa mejor los datos medidos.

Metricas usadas:
- MAE: error absoluto medio
- RMSE: raiz del error cuadratico medio
- IAE: integral del valor absoluto del error
- ISE: integral del error cuadratico

In [ ]:
# Comparacion cuantitativa de error entre modelos y datos
mask_eval = t >= t_step
t_eval = t[mask_eval]
y_eval = y[mask_eval]

def compute_error_metrics(t_arr, y_true, y_pred):
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    if not np.any(valid):
        return {'MAE': np.nan, 'RMSE': np.nan, 'IAE': np.nan, 'ISE': np.nan}

    tt = t_arr[valid]
    e = y_true[valid] - y_pred[valid]

    mae = float(np.mean(np.abs(e)))
    rmse = float(np.sqrt(np.mean(e**2)))
    iae = float(np.trapz(np.abs(e), tt))
    ise = float(np.trapz(e**2, tt))

    return {'MAE': mae, 'RMSE': rmse, 'IAE': iae, 'ISE': ise}

metrics_zn = compute_error_metrics(t_eval, y_eval, y_zn[mask_eval])
metrics_smith = compute_error_metrics(t_eval, y_eval, y_smith[mask_eval])

df_error = pd.DataFrame([
    {'Modelo': 'FOPDT Ziegler-Nichols', **metrics_zn},
    {'Modelo': 'FOPDT Smith', **metrics_smith},
]).set_index('Modelo')

display(df_error.round(4))

# Modelo con menor RMSE e IAE
if np.isfinite(df_error['RMSE']).any():
    best_rmse = df_error['RMSE'].idxmin()
    print(f'Mejor ajuste por RMSE: {best_rmse}')
if np.isfinite(df_error['IAE']).any():
    best_iae = df_error['IAE'].idxmin()
    print(f'Mejor ajuste por IAE: {best_iae}')